# Image recongocnition is just curve fitting ☝️


We are not [cutting edge 1998](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf) 🤓

Time for the canonical first-real-network: an MLP that classifies handwritten digits. Compared to the cubic fit and the bump fit, this notebook brings *everything* in at once:

- **Real data** — 60k training images, 10k test, loaded through `torchvision`.
- **Mini-batch SGD** instead of full-batch.
- **Multi-class classification** — cross-entropy loss over 10 logits.
- **Train / test split** so we can detect overfitting.
- **TensorBoard logging** so we can watch training in real time, in a separate browser tab, without polluting the notebook.

Even though MNIST is small and a 2-layer MLP is tiny by modern standards, the code structure is essentially what `train.py` does in nanoGPT — load batches, forward, loss, backward, optimizer step, log.

### Launching TensorBoard

In a separate terminal:

```sh
./scripts/tensorboard.sh
```

This serves on `0.0.0.0:6006` so it's reachable over Tailscale. Open `http://<this-machine>:6006/` in a browser, then start training below — TensorBoard auto-refreshes.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime as dt

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import transforms
from tqdm.notebook import tqdm

pio.renderers.default = "notebook"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")
if device == "cuda":
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

## The data

MNIST: 70k grayscale 28×28 images of handwritten digits, split into 60k train and 10k test, each labeled 0–9. `torchvision.datasets.MNIST` downloads it once into `./data/` and caches it from then on.

We normalize pixel values to roughly zero mean / unit variance using the standard MNIST stats (mean ≈ 0.1307, std ≈ 0.3081). The normalization is what most published benchmarks use, so our numbers will be comparable.

In [ ]:
from plotly.subplots import make_subplots

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transform)

print(f"Train: {len(train_ds):>5} images")
print(f"Test:  {len(test_ds):>5} images")

# A 4×4 grid of training samples, each rendered as a Heatmap with a 1-px gap
# between cells so every individual pixel is visible.
n_show, ncols = 16, 4
nrows = n_show // ncols

sample_fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[f"label: {train_ds[i][1]}" for i in range(n_show)],
    horizontal_spacing=0.04, vertical_spacing=0.07,
)
for i in range(n_show):
    img = (train_ds[i][0].squeeze().numpy() * MNIST_STD) + MNIST_MEAN
    r, c = i // ncols + 1, i % ncols + 1
    sample_fig.add_trace(
        go.Heatmap(z=img, colorscale="gray", zmin=0.0, zmax=1.0,
                   showscale=False, xgap=1, ygap=1, hoverinfo="skip"),
        row=r, col=c,
    )

# Hide tick labels and gridlines; flip Y so the digit isn't upside down.
sample_fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
sample_fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False,
                        autorange="reversed", scaleanchor="x")
sample_fig.update_layout(
    title="MNIST training samples — thin lines between cells mark individual pixels",
    width=600, height=620,
    margin=dict(l=10, r=10, t=60, b=10),
)
sample_fig.show()

## The model

A two-hidden-layer MLP: **784 → 256 → 64 → 10**, with ReLU between hidden layers.

The output is *raw logits* — we don't apply softmax inside the model. `nn.CrossEntropyLoss` expects logits (it applies `log_softmax` internally), which is numerically more stable than doing softmax + NLL ourselves.

The input has shape `(batch, 1, 28, 28)` (a batch of 1-channel images). We flatten it to `(batch, 784)` inside `forward`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=(256, 64), n_classes=10):
        super().__init__()
        dims = [28 * 28, *hidden, n_classes]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:  # no activation after the last (logit) layer
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, 1, 28, 28) -> (B, 784)
        return self.net(x.view(x.size(0), -1))


model = MLP().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

In [ ]:
import math

# Architecture diagram: one rectangle per layer, sized by log(neurons) for
# visual clarity. Drawing 784 + 256 + 64 + 10 individual circles would be
# unreadable, so we use blocks labeled with their sizes — same style of
# diagram you'd see in a paper.

# Extract layer sizes directly from the model so this stays correct if MLP
# gets reshaped later.
mlp_sizes = []
for layer in model.net:
    if isinstance(layer, nn.Linear):
        if not mlp_sizes:
            mlp_sizes.append(layer.in_features)
        mlp_sizes.append(layer.out_features)
layer_roles = ["input"] + ["hidden"] * (len(mlp_sizes) - 2) + ["output"]
arrow_labels = (["Linear → ReLU"] * (len(mlp_sizes) - 2)) + ["Linear (logits)"]

heights = [math.log10(s) * 0.45 + 0.45 for s in mlp_sizes]
colors  = ["dodgerblue"] + ["seagreen"] * (len(mlp_sizes) - 2) + ["crimson"]

arch_fig = go.Figure()
for i, (size, role, h, color) in enumerate(zip(mlp_sizes, layer_roles, heights, colors)):
    arch_fig.add_shape(
        type="rect",
        x0=i - 0.28, x1=i + 0.28, y0=-h / 2, y1=h / 2,
        fillcolor=color, opacity=0.55,
        line=dict(color="black", width=1.5),
    )
    arch_fig.add_annotation(
        x=i, y=0, text=f"<b>{size}</b>", showarrow=False,
        font=dict(size=20, color="black"),
    )
    arch_fig.add_annotation(
        x=i, y=h / 2 + 0.22, text=role, showarrow=False,
        font=dict(size=12, color="gray"),
    )

# Arrows + operation labels between layers
for i in range(len(mlp_sizes) - 1):
    arch_fig.add_annotation(
        x=i + 1 - 0.28, y=0, ax=i + 0.28, ay=0,
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=2, arrowsize=1.6, arrowwidth=2,
        arrowcolor="dimgray",
    )
    arch_fig.add_annotation(
        x=i + 0.5, y=-(max(heights) / 2 + 0.4),
        text=arrow_labels[i], showarrow=False,
        font=dict(size=11, color="dimgray"),
    )

arch_fig.update_layout(
    title=f"MLP architecture — {n_params:,} parameters",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
               range=[-0.55, len(mlp_sizes) - 0.45]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
               range=[-max(heights) - 0.4, max(heights) / 2 + 0.6]),
    plot_bgcolor="white",
    width=880, height=320,
    margin=dict(l=10, r=10, t=50, b=10),
    showlegend=False,
)
arch_fig.show()

## From curve fitting to classification

Every loss we've used so far has been **MSE** — the squared distance between a *predicted number* and a *true number*. That's regression: predict a real-valued $y$ from input $x$.

MNIST isn't quite the same. The label is a **discrete class** (one of 10 digits), not a number we want to numerically match. "Predicting 5 when the answer is 8" isn't *less wrong* than "predicting 3 when the answer is 8" — they're both just *wrong*. MSE no longer fits.

The reframe: treat the model's output as a **probability distribution** over the 10 classes and measure how far that distribution is from the (known) one-hot target.

### Step 1: raw output

Our MLP's final layer is `nn.Linear(64, 10)`. With no activation on top, it produces **10 unbounded real numbers** — called **logits**. They aren't probabilities yet: they can be negative, they don't sum to anything in particular, they have no scale.

### Step 2: softmax

To turn the 10 logits into a valid probability distribution we apply **softmax**:

$$p_k \;=\; \frac{e^{z_k}}{\sum_{j=0}^{9} e^{z_j}}$$

It exponentiates each logit (forcing positivity) and normalizes by the total (forcing the result to sum to 1). What was 10 arbitrary real numbers is now a probability distribution over the 10 classes.

The cell below runs one test image through our **untrained** MLP and shows both stages of the output.

In [ ]:
# Forward one test image through the model.
# (At this point in the notebook the model is still randomly initialized, so the
# logits will be small and the softmax distribution roughly uniform.)
sample_idx = 0
sample_img, sample_label = test_ds[sample_idx]
sample_img_t = sample_img.unsqueeze(0).to(device)   # (1, 1, 28, 28)

model.eval()
with torch.no_grad():
    logits = model(sample_img_t).cpu().numpy().flatten()  # shape (10,)

# Manual softmax (PyTorch also has F.softmax; doing it by hand here for clarity)
probs = np.exp(logits) / np.exp(logits).sum()
top_class = int(probs.argmax())

img_disp = (sample_img.squeeze().numpy() * MNIST_STD) + MNIST_MEAN

class_fig = make_subplots(
    rows=1, cols=3,
    column_widths=[0.22, 0.39, 0.39],
    specs=[[{"type": "heatmap"}, {"type": "bar"}, {"type": "bar"}]],
    subplot_titles=(f"Input  (true: {sample_label})",
                    "Logits (raw output)",
                    "Softmax (probabilities)"),
    horizontal_spacing=0.06,
)
class_fig.add_trace(
    go.Heatmap(z=img_disp, colorscale="gray", zmin=0, zmax=1, showscale=False,
               xgap=1, ygap=1, hoverinfo="skip"),
    row=1, col=1,
)
class_fig.add_trace(
    go.Bar(x=list(range(10)), y=logits,
           marker_color="lightsteelblue",
           hovertemplate="class %{x}<br>logit %{y:.3f}<extra></extra>"),
    row=1, col=2,
)
class_fig.add_trace(
    go.Bar(x=list(range(10)), y=probs,
           marker_color=["steelblue" if i == top_class else "lightsteelblue"
                         for i in range(10)],
           text=[f"{p:.1%}" for p in probs], textposition="outside",
           hovertemplate="class %{x}<br>p = %{y:.4f}<extra></extra>"),
    row=1, col=3,
)

class_fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=1, col=1)
class_fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False,
                       autorange="reversed", scaleanchor="x", row=1, col=1)
for c in (2, 3):
    class_fig.update_xaxes(title="class", tickmode="linear", tick0=0, dtick=1, row=1, col=c)
class_fig.update_yaxes(title="logit", row=1, col=2)
class_fig.update_yaxes(title="probability", range=[0, max(probs) * 1.3], row=1, col=3)
class_fig.update_layout(width=1000, height=380, template="plotly_white",
                        margin=dict(l=10, r=10, t=50, b=10), showlegend=False)
class_fig.show()

print(f"True class:          {sample_label}")
print(f"Argmax of logits:    {int(logits.argmax())}   ← model's current 'prediction'")
print(f"All logits:          [{', '.join(f'{l:+.2f}' for l in logits)}]")
print()
print(f"Softmax probs sum:   {probs.sum():.6f}   (≈ 1.0, as expected)")
print(f"Softmax distribution is roughly uniform (~{1/10:.0%} per class) — the model")
print("hasn't learned anything yet. After training, one bar should dominate the others.")

## The target distribution, and cross-entropy as the loss

For a single training example with true class $y^*$, the **target distribution** has all of its mass on that one class:

$$t_k \;=\; \begin{cases} 1 & \text{if } k = y^* \\ 0 & \text{otherwise} \end{cases}$$

This is called a **one-hot** vector. For an image with `true label = 7`, the target is `[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]`.

Now we need a way to measure how far the predicted distribution $\mathbf{p}$ is from the target $\mathbf{t}$. The standard choice is **cross-entropy**:

$$\mathcal{L}(\mathbf{p}, \mathbf{t}) \;=\; -\sum_{k} t_k \log p_k$$

Because $t_k$ is 0 everywhere except at the true class, the sum collapses to a single term:

$$\mathcal{L} \;=\; -\log p_{y^*}$$

In plain English: **the loss is the negative log of the probability the model assigned to the correct class**.

- If the model assigns 100% probability to the correct class, $-\log 1 = 0$ — perfect.
- 50% probability → $-\log 0.5 \approx 0.69$.
- A random untrained model on 10 classes sits near uniform, so its loss is around $-\log 0.1 \approx 2.30$.
- As $p_{y^*} \to 0$, the loss $\to +\infty$. The model is *strongly* punished for being confidently wrong.

PyTorch's `nn.CrossEntropyLoss` takes the **raw logits** (not the softmax outputs) and an integer label. Internally it does log-softmax + negative-log-likelihood in one numerically stable step. The cell below verifies our manual calculation matches.

In [ ]:
# Build the one-hot target distribution for the example we forwarded above.
target = np.zeros(10)
target[sample_label] = 1.0

# Cross-entropy two ways — manual and PyTorch — to show they agree.
ce_manual = float(-np.log(probs[sample_label]))

logits_t = torch.tensor(logits).unsqueeze(0).to(device)   # (1, 10)
label_t  = torch.tensor([sample_label], device=device)    # (1,)
ce_torch = nn.CrossEntropyLoss()(logits_t, label_t).item()

# Visualize predicted distribution vs. target side-by-side
ce_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Predicted distribution (softmax)",
                    f"Target distribution (one-hot at class {sample_label})"),
    horizontal_spacing=0.08,
)
ce_fig.add_trace(
    go.Bar(x=list(range(10)), y=probs,
           marker_color=["steelblue" if i == probs.argmax() else "lightsteelblue"
                         for i in range(10)],
           text=[f"{p:.1%}" for p in probs], textposition="outside"),
    row=1, col=1,
)
ce_fig.add_trace(
    go.Bar(x=list(range(10)), y=target,
           marker_color=["crimson" if i == sample_label else "lightgray"
                         for i in range(10)],
           text=[f"{t:.0f}" for t in target], textposition="outside"),
    row=1, col=2,
)
for c in (1, 2):
    ce_fig.update_xaxes(title="class", tickmode="linear", tick0=0, dtick=1, row=1, col=c)
    ce_fig.update_yaxes(title="probability", range=[0, 1.15], row=1, col=c)
ce_fig.update_layout(width=950, height=380, template="plotly_white",
                     margin=dict(l=10, r=10, t=50, b=10), showlegend=False)
ce_fig.show()

print(f"True class:                              {sample_label}")
print(f"Model's probability for the true class:  p[{sample_label}] = {probs[sample_label]:.4f}")
print()
print(f"Cross-entropy (manual)   = -log(p[{sample_label}]) = {ce_manual:.4f}")
print(f"Cross-entropy (PyTorch)  = nn.CrossEntropyLoss(logits, label) = {ce_torch:.4f}")
print()
print(f"Sanity check: a random model on {10} classes should sit around -log(1/10) = "
      f"{-np.log(1/10):.4f}.")

## Side-by-side: regression vs. classification

| | **Curve fitting (regression)** | **Classification** |
| --- | --- | --- |
| Target | a real number per input | a class label (integer 0…K−1) |
| Model output | one real number | $K$ raw **logits** (one per class) |
| Activation on output | none (linear) | **softmax** for probability interpretation |
| Target as used in the loss | the number itself | **one-hot** vector |
| Loss function | **MSE**: $(\hat{y} - y)^2$ | **Cross-entropy**: $-\log p_{y^*}$ |
| Loss = 0 when… | $\hat{y} = y$ | $p_{y^*} = 1$ |
| How loss grows when wrong | quadratically in the error | sharply: $-\log p_{y^*} \to \infty$ as $p_{y^*} \to 0$ |
| Final prediction (inference) | the network's output | $\arg\max_k p_k$ (equivalently $\arg\max_k z_k$) |
| Example tasks | predict price, temperature, position | digit recognition, sentiment, next-token in a sentence |

**What stays the same.** The model architecture (`nn.Linear` + activations), the forward pass, autograd, the optimizer, the dataloader, the training loop structure — all identical.

**What changes.** Only the *size of the output layer* (1 for regression, $K$ for classification) and the *loss function* (MSE vs. cross-entropy).

That's why nanoGPT's `train.py` looks structurally like notebooks 2 and 3: GPT is predicting the *next token* — a class out of ~50k. The surrounding machinery is everything we've already built.

## The training loop

Each training step is exactly the four-step ritual from `2_pytorch_fit.ipynb`, now wrapped in two loops — one over epochs, one over mini-batches:

1. forward, compute loss
2. `loss.backward()`
3. `optimizer.step()`
4. `optimizer.zero_grad()`

Every run gets its own subdirectory under `runs/` named with a timestamp so you can compare multiple runs side-by-side in TensorBoard. We log:

- **`train/loss`** — every batch (fine-grained noisy curve)
- **`train/epoch_loss`**, **`val/loss`**, **`val/accuracy`** — once per epoch
- **`samples/train`** — a batch of images, so you can verify they look right after normalization
- The **computation graph** (`add_graph`), visible in TensorBoard's "Graphs" tab

If you don't have TensorBoard open yet, run `./scripts/tensorboard.sh` in another terminal and refresh the browser tab — the curves appear in real time as training progresses.

In [ ]:
# ---- hyperparameters ----
batch_size = 128
n_epochs   = 5
lr         = 1e-3

# ---- data loaders ----
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=(device == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=512,        shuffle=False, num_workers=2, pin_memory=(device == "cuda"))

# ---- fresh model / optimizer / logger ----
model     = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn   = nn.CrossEntropyLoss()

run_name = f"mlp-h256-64-bs{batch_size}-{dt.datetime.now().strftime('%Y%m%d-%H%M%S')}"
log_dir  = f"runs/{run_name}"
writer   = SummaryWriter(log_dir)
print(f"Logging to: {log_dir}")

# Log a batch of (un-normalized) samples and the model graph once at step 0
with torch.no_grad():
    sample_batch, _ = next(iter(train_loader))
    writer.add_images("samples/train", sample_batch[:16] * MNIST_STD + MNIST_MEAN, 0)
    writer.add_graph(model, sample_batch.to(device))

# ---- in-memory loss history (so we can plot it inline below) ----
batch_train_losses  = []
epoch_train_losses  = []
epoch_val_losses    = []
epoch_val_accs      = []

# ---- training loop ----
global_step = 0
for epoch in range(n_epochs):
    # train
    model.train()
    epoch_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch + 1}/{n_epochs}", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_train_losses.append(loss.item())
        writer.add_scalar("train/loss", loss.item(), global_step)
        global_step += 1
        epoch_loss += loss.item() * images.size(0)
    epoch_loss /= len(train_ds)
    epoch_train_losses.append(epoch_loss)
    writer.add_scalar("train/epoch_loss", epoch_loss, epoch)

    # evaluate
    model.eval()
    val_loss = 0.0
    correct  = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            val_loss += loss_fn(logits, labels).item() * images.size(0)
            correct  += (logits.argmax(dim=1) == labels).sum().item()
    val_loss /= len(test_ds)
    val_acc   = correct / len(test_ds)
    epoch_val_losses.append(val_loss)
    epoch_val_accs.append(val_acc)
    writer.add_scalar("val/loss", val_loss, epoch)
    writer.add_scalar("val/accuracy", val_acc, epoch)

    print(f"epoch {epoch + 1}:  train_loss={epoch_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

writer.close()
print(f"\nDone. Logs in {log_dir}")

In [ ]:
# Plot the training/validation loss curves we collected above.
# Per-batch train loss is noisy (one mini-batch ≠ the whole train set), so we
# show it as a light line and overlay the per-epoch means heavily.

# x-axis: convert batch index into "fractional epoch"
batches_per_epoch = len(batch_train_losses) // n_epochs
batch_x = (np.arange(len(batch_train_losses)) + 1) / batches_per_epoch
epoch_x = np.arange(1, n_epochs + 1)

loss_fig = go.Figure()
loss_fig.add_trace(go.Scatter(
    x=batch_x, y=batch_train_losses, mode="lines",
    name="train (per batch)",
    line=dict(color="lightsteelblue", width=1), opacity=0.7,
))
loss_fig.add_trace(go.Scatter(
    x=epoch_x, y=epoch_train_losses, mode="lines+markers",
    name="train (per epoch)",
    line=dict(color="steelblue", width=3), marker=dict(size=10),
))
loss_fig.add_trace(go.Scatter(
    x=epoch_x, y=epoch_val_losses, mode="lines+markers",
    name="validation (per epoch)",
    line=dict(color="crimson", width=3), marker=dict(size=10, symbol="diamond"),
))
loss_fig.update_layout(
    title="Training and validation loss",
    xaxis_title="epoch",
    yaxis_title="cross-entropy loss",
    template="plotly_white",
    width=820, height=420,
    margin=dict(l=10, r=10, t=50, b=10),
)
loss_fig.show()

print(f"Final train loss: {epoch_train_losses[-1]:.4f}")
print(f"Final val   loss: {epoch_val_losses[-1]:.4f}")
print(f"Final val accuracy: {epoch_val_accs[-1] * 100:.2f}%")

## What did the network actually learn?

Pick a batch from the test set, run it through the model, and visualize predictions. Green title = correct, red = wrong.

In [ ]:
model.eval()
test_images, test_labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(test_images.to(device)).argmax(dim=1).cpu()

n_show = 16
imgs_display = (test_images[:n_show].squeeze(1).numpy() * MNIST_STD) + MNIST_MEAN

fig = px.imshow(imgs_display, facet_col=0, facet_col_wrap=4,
                color_continuous_scale="gray", binary_string=True,
                width=560, height=600,
                title="Test predictions (green=correct, red=wrong)")
for i in range(n_show):
    pred, true = preds[i].item(), test_labels[i].item()
    color = "green" if pred == true else "red"
    fig.layout.annotations[i].text = f"<span style='color:{color}'>pred={pred} (true={true})</span>"
fig.update_layout(coloraxis_showscale=False, margin=dict(l=10, r=10, t=40, b=10))
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

## Confusion: which digits confuse the model most?

Run the whole test set through the model and tally a 10×10 confusion matrix. Cell $(i, j)$ counts how many times *true class $i$* was *predicted as $j$*. The diagonal should dominate.

In [ ]:
confusion = np.zeros((10, 10), dtype=int)

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(dim=1).cpu().numpy()
        for t, p in zip(labels.numpy(), preds):
            confusion[t, p] += 1

cm_fig = go.Figure(data=go.Heatmap(
    z=confusion,
    x=list(range(10)),
    y=list(range(10)),
    colorscale="Blues",
    text=confusion, texttemplate="%{text}",
    colorbar=dict(title="count"),
))
cm_fig.update_layout(
    title=f"Confusion matrix on the {len(test_ds)} test images",
    xaxis_title="predicted", yaxis_title="true",
    yaxis_autorange="reversed",
    width=620, height=560,
    template="plotly_white",
)
cm_fig.show()

overall_acc = np.trace(confusion) / confusion.sum()
print(f"Overall test accuracy: {overall_acc:.4f}")
worst_class = np.argmin(np.diag(confusion) / confusion.sum(axis=1))
print(f"Worst-fit digit: {worst_class}  "
      f"(per-class acc {confusion[worst_class, worst_class] / confusion[worst_class].sum():.3f})")

## Try it yourself: draw a digit

Draw a digit with your mouse on the canvas below — the model classifies it live. Some practical notes:

- The model was trained on **MNIST**, where digits are centered, white-on-black, and ~20 pixels tall in a 28×28 frame. So **draw a thick digit roughly centered** in the canvas for best results.
- After each stroke (mouse-up) the canvas is downsampled to 28×28, re-centered by center-of-mass, normalized with the MNIST stats, and fed through the model. The bar chart shows the softmax probabilities over the ten classes.
- You'll notice the model is **a lot less confident on hand-drawn digits than on MNIST test images** — partly because real MNIST digits look surprisingly *machine-y* (very thin strokes, consistent style), and partly because an MLP doesn't see images the way a CNN does. Try drawing a "7" with a horizontal bar across it: many people draw it that way, but MNIST has no such examples, so the model gets confused.

If the cell errors with `ModuleNotFoundError: No module named 'ipycanvas'`, run `uv sync --extra notebooks` and restart the kernel.

In [ ]:
import io
import ipywidgets as widgets
from ipycanvas import Canvas
from PIL import Image as PILImage

# ---- the canvas ----
CANVAS_PX = 280   # display size (10x MNIST for easy drawing)
BRUSH_PX  = 22    # thick strokes so they survive downsampling to 28x28

# sync_image_data=True makes the frontend push PNG-encoded pixel bytes back to
# the kernel as the canvas changes. We read `draw_canvas.image_data` (bytes) and
# decode it ourselves with PIL — this avoids the racy `get_image_data()` path.
draw_canvas = Canvas(width=CANVAS_PX, height=CANVAS_PX, sync_image_data=True)

def reset_canvas():
    draw_canvas.fill_style = "black"
    draw_canvas.fill_rect(0, 0, CANVAS_PX, CANVAS_PX)

reset_canvas()
draw_canvas.stroke_style = "white"
draw_canvas.line_width   = BRUSH_PX
draw_canvas.line_cap     = "round"
draw_canvas.line_join    = "round"

_drawing = {"on": False}

def on_mouse_down(x, y):
    _drawing["on"] = True
    draw_canvas.begin_path()
    draw_canvas.move_to(x, y)

def on_mouse_move(x, y):
    if _drawing["on"]:
        draw_canvas.line_to(x, y)
        draw_canvas.stroke()

def on_mouse_up(x, y):
    _drawing["on"] = False
    # auto-predict; may use slightly stale pixels (frontend->kernel sync is async)
    try:
        predict()
    except Exception as e:
        label.value = f"<h3 style='margin:0;color:#a33'>auto-predict failed: {e}</h3>"

draw_canvas.on_mouse_down(on_mouse_down)
draw_canvas.on_mouse_move(on_mouse_move)
draw_canvas.on_mouse_up(on_mouse_up)


# ---- output widgets ----
label = widgets.HTML(value="<h3 style='margin:0'>Draw a digit and hit <b>Predict</b> →</h3>")

prob_fig = go.FigureWidget()
prob_fig.add_trace(go.Bar(
    x=list(range(10)),
    y=[0.0] * 10,
    marker=dict(color=["lightsteelblue"] * 10),
    hovertemplate="digit %{x}<br>prob %{y:.3f}<extra></extra>",
))
prob_fig.update_layout(
    title="Softmax probabilities",
    xaxis=dict(title="digit", tickmode="linear", tick0=0, dtick=1),
    yaxis=dict(title="probability", range=[0, 1]),
    template="plotly_white",
    width=480, height=320,
    margin=dict(l=40, r=20, t=40, b=40),
)


def center_by_mass(arr):
    """Shift a grayscale 28x28 array so the digit's center of mass is at (14, 14)."""
    if arr.sum() < 1.0:
        return arr
    h, w = arr.shape
    ys, xs = np.indices(arr.shape)
    cy = (ys * arr).sum() / arr.sum()
    cx = (xs * arr).sum() / arr.sum()
    return np.roll(arr, shift=(int(round(h / 2 - cy)), int(round(w / 2 - cx))), axis=(0, 1))


def predict():
    raw = draw_canvas.image_data  # PNG-encoded bytes pushed by the frontend
    if not raw:
        label.value = ("<h3 style='margin:0;color:#a33'>no pixels yet — draw something, "
                       "then click <b>Predict</b></h3>")
        return

    # Decode PNG -> grayscale numpy
    pil = PILImage.open(io.BytesIO(raw)).convert("L")
    arr_full = np.asarray(pil, dtype=np.float32) / 255.0  # (CANVAS_PX, CANVAS_PX)

    # Downsample to 28x28 (MNIST resolution)
    arr28 = np.asarray(
        PILImage.fromarray((arr_full * 255).astype(np.uint8))
            .resize((28, 28), PILImage.LANCZOS),
        dtype=np.float32,
    ) / 255.0
    arr28 = center_by_mass(arr28)

    normalized = (arr28 - MNIST_MEAN) / MNIST_STD
    tensor = torch.from_numpy(normalized).float().unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy().flatten()

    top = int(probs.argmax())
    colors = ["lightsteelblue"] * 10
    colors[top] = "steelblue"
    with prob_fig.batch_update():
        prob_fig.data[0].y = probs
        prob_fig.data[0].marker.color = colors
    label.value = (f"<h3 style='margin:0'>Prediction: <b>{top}</b> "
                   f"&nbsp; <span style='color:#888'>({probs[top]:.1%} confident)</span></h3>")


# ---- buttons ----
half_width  = f"{CANVAS_PX // 2 - 2}px"
predict_btn = widgets.Button(description="Predict", button_style="primary",
                             icon="play", layout=widgets.Layout(width=half_width))
clear_btn   = widgets.Button(description="Clear", button_style="warning",
                             icon="trash", layout=widgets.Layout(width=half_width))

def on_predict(_):
    try:
        predict()
    except Exception as e:
        label.value = f"<h3 style='margin:0;color:#a33'>predict failed: {e}</h3>"

def on_clear(_):
    reset_canvas()
    with prob_fig.batch_update():
        prob_fig.data[0].y = [0.0] * 10
        prob_fig.data[0].marker.color = ["lightsteelblue"] * 10
    label.value = "<h3 style='margin:0'>Draw a digit and hit <b>Predict</b> →</h3>"

predict_btn.on_click(on_predict)
clear_btn.on_click(on_clear)

buttons_row = widgets.HBox([predict_btn, clear_btn],
                           layout=widgets.Layout(width=f"{CANVAS_PX}px"))
left_panel  = widgets.VBox([draw_canvas, buttons_row])
right_panel = widgets.VBox([label, prob_fig])
display(widgets.HBox([left_panel, right_panel]))

## Next steps

Things to try yourself. Each becomes a *separate run* in TensorBoard (the `run_name` is timestamped, and you can also tag it explicitly), so you can compare curves side-by-side in the left panel:

- **Width / depth**: `MLP(hidden=(512, 256, 128))` — more capacity, but does it overfit?
- **Optimizer**: replace `Adam` with `torch.optim.SGD(..., lr=0.05, momentum=0.9)`. Watch the train loss curve get noisier.
- **Activation**: swap `nn.ReLU()` for `nn.GELU()` or `nn.Tanh()`.
- **Regularization**: add `nn.Dropout(0.2)` between layers; or `weight_decay=1e-4` to the optimizer.
- **Learning rate schedule**: wrap the optimizer in `torch.optim.lr_scheduler.CosineAnnealingLR(...)` and call `scheduler.step()` once per epoch.
- **Data augmentation**: insert `transforms.RandomAffine(degrees=10, translate=(0.1, 0.1))` before `ToTensor` and re-train.

To compare in TensorBoard, modify the `run_name` line with a tag, e.g.:

```python
run_name = f"mlp-{tag}-{dt.datetime.now().strftime('%H%M%S')}"
```

We've now used every conceptual block needed to read `train.py` in nanoGPT: a model as `nn.Module`, batched data loading via `DataLoader`, autograd, an optimizer step, and logging. From here, scaling up to a transformer is mainly about model architecture (see `model.py` in this repo).